# IM-RCS fitted distribution over preference lists

This notebook reads the **50 canonical IM-RCS fits** from `sequential_rcs/results/fits/` and produces only `sequential_rcs/results/overall_beta.pdf`. Bars show the arithmetic mean of each fitted ranking weight, and error bars show the sample standard deviation across the five sites and ten seeded trials. Legacy `opt_data` results are not used.

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')


def find_repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'sequential_rcs' / 'results' / 'fits').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate sequential_rcs/results/fits from the current directory.')


ROOT = find_repo_root()
FIT_DIR = ROOT / 'sequential_rcs' / 'results' / 'fits'
FIG_PATH = ROOT / 'sequential_rcs' / 'results' / 'overall_beta.pdf'

SITES = (5, 14, 15, 24, 32)
TRIALS = tuple(range(1, 11))
RANK_META = {
    'position_asc': ('position (top->down)', 'position'),
    'price_low': ('price low->high', 'price'),
    'price_high': ('price high->low', 'price'),
    'star_high': ('star high->low', 'star'),
    'star_low': ('star low->high', 'star'),
    'location_high': ('loc high->low', 'location'),
    'location_low': ('loc low->high', 'location'),
    'review_high': ('review high->low', 'review'),
    'review_low': ('review low->high', 'review'),
}
ATTR_COLOR = {
    'position': '#4C72B0',
    'star': '#C44E52',
    'price': '#55A868',
    'review': '#CCB974',
    'location': '#8172B3',
}
print('fit directory:', FIT_DIR)
print('output figure:', FIG_PATH)

In [ ]:
def read_matlab_char(h5, ref):
    values = np.asarray(h5[ref]).reshape(-1)
    return ''.join(chr(int(value)) for value in values)


def read_fit(path):
    with h5py.File(path, 'r') as h5:
        name_refs = np.asarray(h5['artifact/model/rankings/name']).reshape(-1)
        names = [read_matlab_char(h5, ref) for ref in name_refs]
        beta = np.asarray(h5['artifact/model/beta'], dtype=float).reshape(-1)
        site = int(np.asarray(h5['artifact/site']).squeeze())
        trial = int(np.asarray(h5['artifact/trial']).squeeze())
    if names != list(RANK_META):
        raise AssertionError(f'Unexpected ranking order in {path.name}: {names}')
    if beta.shape != (9,) or not np.isfinite(beta).all() or (beta < 0).any():
        raise AssertionError(f'Invalid beta vector in {path.name}')
    if not np.isclose(beta.sum(), 1.0, atol=1e-10):
        raise AssertionError(f'Beta does not sum to one in {path.name}: {beta.sum()}')
    return site, trial, beta


files = sorted(FIT_DIR.glob('fit_*_*_imrcs.mat'))
if len(files) != 50:
    raise AssertionError(f'Expected 50 IM-RCS artifacts, found {len(files)}.')

records = []
for fit_path in files:
    site, trial, beta = read_fit(fit_path)
    records.append({'site': site, 'trial': trial, **dict(zip(RANK_META, beta))})

df = pd.DataFrame(records).sort_values(['site', 'trial']).reset_index(drop=True)
expected_keys = {(site, trial) for site in SITES for trial in TRIALS}
actual_keys = set(zip(df['site'], df['trial']))
if actual_keys != expected_keys or df.duplicated(['site', 'trial']).any():
    raise AssertionError('The 5-site x 10-trial IM-RCS grid is incomplete or duplicated.')
if not np.allclose(df[list(RANK_META)].sum(axis=1), 1.0, atol=1e-10):
    raise AssertionError('At least one fitted beta vector does not sum to one.')

print(f'validated {len(df)} fits: {len(SITES)} sites x {len(TRIALS)} trials')

In [ ]:
overall = df[list(RANK_META)].agg(['mean', 'std']).T
overall['label'] = [RANK_META[name][0] for name in overall.index]
overall['group'] = [RANK_META[name][1] for name in overall.index]
overall = overall.sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(9.5, 4.6))
x = np.arange(len(overall))
colors = [ATTR_COLOR[group] for group in overall['group']]
ax.bar(
    x,
    overall['mean'],
    yerr=overall['std'],
    capsize=3,
    color=colors,
    alpha=0.9,
)
ax.set_xticks(x)
ax.set_xticklabels(overall['label'], rotation=35, ha='right', fontsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.set_ylabel('Probability', fontsize=14)
ax.set_title('Fitted distribution over preference lists', fontsize=15)

for i, (mean, std) in enumerate(zip(overall['mean'], overall['std'])):
    value_label = f'{mean:.3f}' if mean >= 1e-3 else f'{mean:.1e}'
    ax.text(i, mean + std + 0.008, value_label, ha='center', fontsize=11)

upper = min(1.05, float((overall['mean'] + overall['std']).max() + 0.08))
ax.set_ylim(0, upper)
legend_order = ('position', 'star', 'price', 'review', 'location')
ax.legend(
    handles=[Patch(color=ATTR_COLOR[group], label=group) for group in legend_order],
    fontsize=11,
    ncol=5,
    loc='upper right',
)
fig.tight_layout()
fig.savefig(FIG_PATH, bbox_inches='tight')
plt.show()

print('saved ->', FIG_PATH)
overall[['mean', 'std']]